In [ ]:
from IPython.display import clear_output

%pip install kagglehub catboost xgboost tqdm -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
from tqdm import tqdm

%matplotlib inline

clear_output()

In [ ]:


# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")


print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:

csv_path = os.path.join(path, "Q3_data.csv")

df = pd.read_csv(csv_path)
df.head()

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:

df.describe()

In [ ]:
# Task 1: Write your code here:


missing_pct = df.isna().mean().sort_values(ascending=False)
missing_pct.head(10)


threshold = 0
cols_to_drop = missing_pct[missing_pct > threshold].index
df = df.drop(columns=cols_to_drop)



In [ ]:
# Task 2: Write your code here:

def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Task 3: Write your code here:

#There is no categorical column here so there is no need of encoding


In [ ]:
# Task 4: Write your code here:

from sklearn.preprocessing import StandardScaler

features = df.columns.drop("Target")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
df[features] = scaler.fit_transform(df[features])
df.head()



In [ ]:
# Task 5: Write your code here:

def check_target_imbalance(df, target_column):
  print("Target Distribution:")
  print(df[target_column].value_counts(normalize=True))
  sns.countplot(x=df[target_column])
  plt.title("Target Distribution")
  plt.show()

check_target_imbalance(df, "Target")

In [ ]:
# Task 1: Write your code here:

X = df.drop("Target", axis=1).astype(float)
y = df['Target'].astype(float)

In [ ]:
# Import models

from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold

In [ ]:
sklearn_models = {
  "CatBoost": CatBoostClassifier(
      verbose=0,
      n_estimators=320,
      max_depth=4
  )
}

In [ ]:
all_results = {}

for name in sklearn_models:
  all_results[name] = {'accuracy': [], 'precision': [], 'recall': [], 'f1': []}

In [ ]:
n_splits = 5 # K

# Stratified 5-Fold Cross-Validation, shuffled
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

In [ ]:
for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  # 1. Split data
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # 2. Train & Validate sklearn models
  for model_name, model in sklearn_models.items():
    print(f"Training {model_name}...")
    model.fit(X_train, y_train) # train
    y_pred = model.predict(X_test) # validate

    # 3. Save metrics for that model in this fold
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    all_results[model_name]['accuracy'].append(accuracy)
    all_results[model_name]['precision'].append(precision)
    all_results[model_name]['recall'].append(recall)
    all_results[model_name]['f1'].append(f1)

In [ ]:
for model_name in all_results:
  print(f"\n{model_name}:")
  print(f"  Accuracy:  {np.mean(all_results[model_name]['accuracy']):.4f}")
  print(f"  Precision: {np.mean(all_results[model_name]['precision']):.4f}")
  print(f"  Recall:    {np.mean(all_results[model_name]['recall']):.4f}")
  print(f"  F1-Score:  {np.mean(all_results[model_name]['f1']):.4f}")

In [ ]:
# Task 1: Write your code here:

# Gather importances from the models (from the last fold)
importances = {}

importances['CatBoost'] = sklearn_models['CatBoost'].feature_importances_

# Create a 1x3 plot
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes = axes.flatten()
features = X.columns

for i, (model_name, imp) in enumerate(importances.items()):
  # Sort features by importance for a cleaner plot
  sorted_idx = np.argsort(imp)

  ax = axes[i]
  ax.barh(features[sorted_idx], imp[sorted_idx])
  ax.set_title(f"{model_name} Feature Importance")
  ax.set_xlabel("Importance Score")

plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:

avg_importance = np.mean(np.vstack(list(importances.values())), axis=0)

golden_idx = np.argmax(avg_importance)
golden_feature = X.columns[golden_idx]

print(f"Overall golden feature: {golden_feature}")


In [ ]:
# Task Bonus: Write your code here:

import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score
from catboost import CatBoostClassifier

# --- set this to your golden feature name ---
GOLDEN_FEATURE = "your_golden_feature_name_here"

# 1) Create new X with only the golden feature
X_golden = X[[GOLDEN_FEATURE]].copy()

# 2) Run the same KFold loop with this single feature
kf = KFold(n_splits=5, shuffle=True, random_state=42)  # match your original settings!

gold_fold_acc = []

for fold, (tr_idx, va_idx) in enumerate(kf.split(X_golden), 1):
    X_tr, X_va = X_golden.iloc[tr_idx], X_golden.iloc[va_idx]
    y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

    model = CatBoostClassifier(
        loss_function="Logloss",
        eval_metric="Accuracy",
        random_seed=42,
        verbose=False,
        # keep the same hyperparams you used for the full model:
        # iterations=..., depth=..., learning_rate=..., l2_leaf_reg=..., etc.
    )

    model.fit(X_tr, y_tr)

    preds = model.predict(X_va)
    acc = accuracy_score(y_va, preds)
    gold_fold_acc.append(acc)

gold_mean = float(np.mean(gold_fold_acc))
gold_std = float(np.std(gold_fold_acc))
